SAM post-processing (as used in [FeatureForest](github.com/juglab/featureforest)) prompts SAM with the bounding box of each region of a given class(es) of a segmentation. It is best used to correct large foreground phases, as opposed to connected matrix phases which SAM struggles with. Here we use an ONNX implementation of [EfficientSAM](https://github.com/yformer/EfficientSAM), which allows us to run the whole thing on the CPU without torch.

In [ ]:
from interactive_seg_backend.file_handling import load_image, load_labels
from interactive_seg_backend.extensions.sam_onnx import SAMEncoderONNX, SAMDecoderONNX
from interactive_seg_backend.configs import ClassInfo, TrainingConfig, FeatureConfig
from interactive_seg_backend.main import featurise, train_and_apply

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from skimage.color import label2rgb

Here's an example of using it manually - we init the encoder and decoder models, generate an embedding, and run the decoder with a series of prompts:

In [ ]:
encoder = SAMEncoderONNX(None)
decoder = SAMDecoderONNX(None)

In [ ]:
image = load_image('../tests/data/0.tif')
labels = load_labels('../tests/data/0_labels.tif')

embed = encoder.get_embedding(image)

In [ ]:
prompt_point = (200, 200)
point_mask, _ = decoder.masks_from_points(embed, image.shape, [prompt_point], None, multimask_output=False)

bbox = (80, 140, 200, 130)
box_mask, _ = decoder.masks_from_boxes(embed, image.shape, [bbox], multimask_output=False)

In [ ]:
fig, axs = plt.subplots(1, 2)

overlaid = label2rgb(point_mask, image=image, alpha=0.25, bg_label=0)
axs[0].imshow(overlaid)
axs[0].plot(prompt_point[0], prompt_point[1], 'ro')
axs[0].set_title('Point Prompt')


overlaid = label2rgb(box_mask, image=image, alpha=0.25, bg_label=0)
axs[1].imshow(overlaid)
axs[1].set_title('Box Prompt')

rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=1, edgecolor='r', facecolor='none')
fig.gca().add_artist(rect)

Here we use the SAM post-processing as implemented in `interactive-seg-backend` - we simply enable `do_sam_postproc` for a class in the `TrainingConfig()` and run `train_and_apply()`

In [ ]:

class_infos = [ClassInfo(name="Secondary precipitate", value=1, do_sam_postproc=True, min_size_px=600)]
tc = TrainingConfig(feature_config=FeatureConfig())
tc_sam = TrainingConfig(feature_config=FeatureConfig(), class_infos=class_infos)

feats = featurise(image, tc)

seg, _, _ = train_and_apply(feats, labels, tc)
seg_with_postproc, _, _ = train_and_apply(feats, labels, tc_sam, image)

In [ ]:
from interactive_seg_backend.utils import class_avg_miou, apply_labels_as_overlay, add_inset_zoom, PALETTE_RGB_NORM
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

gt = load_labels('../tests/data/0_ground_truth.tif')

img_rgb = np.expand_dims(image, -1)
img_with_labels = apply_labels_as_overlay(labels, img_rgb, PALETTE_RGB_NORM[1:])
seg_remapped = label2rgb(seg + 1, bg_label=0, colors=PALETTE_RGB_NORM[1:])
seg_postproc_remapped = label2rgb(seg_with_postproc + 1, bg_label=0, colors=PALETTE_RGB_NORM[1:])

seg_miou = class_avg_miou(seg, gt)
seg_postproc_miou = class_avg_miou(seg_with_postproc, gt)

print(f"Seg mIoU: {seg_miou:.3f}")
print(f"Seg + SAM post-proc mIoU: {seg_postproc_miou:.3f}")

axs[0].imshow(img_with_labels)
axs[0].set_title('Image + labels\n ')
axs[1].imshow(seg_remapped)
axs[1].set_title(f'RF segmentation\n mIoU: {seg_miou:.3f}')
add_inset_zoom(axs[1], xywh=[130, 280, 100, 100], fig_xywh=[0.5, -0.2, 0.4, 0.4], img_arr=seg_remapped, labels=None, colors=PALETTE_RGB_NORM[1:], alpha=0.25)
axs[2].imshow(seg_postproc_remapped)
add_inset_zoom(axs[2], xywh=[130, 280, 100, 100], fig_xywh=[0.5, -0.2, 0.4, 0.4], img_arr=seg_postproc_remapped, labels=None, colors=PALETTE_RGB_NORM[1:], alpha=0.25)
axs[2].set_title(f'+ SAM post-proc\n mIoU: {seg_postproc_miou:.3f}')

for ax in axs:
    ax.axis('off')